### Import libraries

In [43]:
import pandas as pd

### Visualize data

In [44]:
tour_df = pd.read_csv('tour.csv')
display(tour_df.head())

display(tour_df['Tour title'])

tour_df.info()

,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571",[1]
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571",[3]
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294",[6]
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795",[7]
4,5,2[4],NaN,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173",[8]


0                              The Eras Tour †
1                       Renaissance World Tour
2                  Sticky & Sweet Tour ‡[4][a]
3                  Beautiful Trauma World Tour
4                      Reputation Stadium Tour
5                                The MDNA Tour
6                    Taking Chances World Tour
7                            Summer Carnival †
8                     The Formation World Tour
9                          The 1989 World Tour
10             The Mrs. Carter Show World Tour
11                     The Monster Ball Tour *
12                        Prismatic World Tour
13    Living Proof: The Farewell Tour ‡[21][a]
14                            Confessions Tour
15                   The Truth About Love Tour
16                          Born This Way Ball
17                            Rebel Heart Tour
18                             Adele Live 2016
19                                The Red Tour
Name: Tour title, dtype: object

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Rank                              20 non-null     int64 
 1   Peak                              9 non-null      object
 2   All Time Peak                     6 non-null      object
 3   Actual gross                      20 non-null     object
 4   Adjusted gross (in 2022 dollars)  20 non-null     object
 5   Artist                            20 non-null     object
 6   Tour title                        20 non-null     object
 7   Year(s)                           20 non-null     object
 8   Shows                             20 non-null     int64 
 9   Average gross                     20 non-null     object
 10  Ref.                              20 non-null     object
dtypes: int64(2), object(9)
memory usage: 1.8+ KB


-

### Clean column names

In [ ]:

tour_df.columns = tour_df.columns.str.lower()
tour_df.columns = tour_df.columns.str.replace(' ', '_')
tour_df.columns = tour_df.columns.str.replace('\xa0', '_')
tour_df.columns = tour_df.columns.str.strip()


#create a list of columns to clean
columns = ['actual_gross', 'adjusted_gross_(in_2022_dollars)', 'average_gross']

# Clean the specified list columns and convert them to numeric
for col in columns:
    tour_df[col] = tour_df[col].astype(str).str.replace('$', '', regex=False)
    tour_df[col] = tour_df[col].astype(str).str.replace(',', '', regex=False)
    tour_df[col] = pd.to_numeric(tour_df[col], errors='coerce')
    

### Removing the 'ref' Column

The 'ref' column was removed because it contains source citation markers rather than relevant data for the analysis.

In [46]:
tour_df = tour_df.drop(columns=['ref.'])

### Clean the 'peak' and 'all_time_peak' columns

In [ ]:
#Convert the column to strings first
tour_df['peak'] = tour_df['peak'].astype('string')
tour_df['all_time_peak'] = tour_df['all_time_peak'].astype('string')

# Remove the content inside square brackets
tour_df['peak'] = tour_df['peak'].str.replace(r'\[.*?\]', '', regex=True).str.strip()
tour_df['all_time_peak'] = tour_df['all_time_peak'].str.replace(r'\[.*?\]', '', regex=True).str.strip()

# Convert the column to numeric
tour_df['peak'] = pd.to_numeric(tour_df['peak'], errors='coerce').astype('Int64')
tour_df['all_time_peak'] = pd.to_numeric(tour_df['all_time_peak'], errors='coerce').astype('Int64')  


### Format the 'year' column


there are some values in the 'year' column that contain 2 years. the best option is to create a new column called 'year_start' and 'year_end' to store the start and end year of the tour.


In [55]:

#extraxt the years from the 'year(s)' column using regex and create new columns for start and end year
years = tour_df['year(s)'].str.extract(
    r'(\d{4})(?:\D+(\d{4}))?'
)

tour_df['start_year'] = years[0]
tour_df['end_year'] = years[1].fillna(years[0])  # Fill NaN with start year if end year is missing


In [49]:
# Clean the tour_title column by removing anything that is not text

display(tour_df['tour_title'])
tour_df['tour_title'] = tour_df['tour_title'].str.extract(r'([a-zA-Z0-9\s]+)')

display(tour_df['tour_title']) #now the tour_title column is cleaned and only contains text



0                              The Eras Tour †
1                       Renaissance World Tour
2                  Sticky & Sweet Tour ‡[4][a]
3                  Beautiful Trauma World Tour
4                      Reputation Stadium Tour
5                                The MDNA Tour
6                    Taking Chances World Tour
7                            Summer Carnival †
8                     The Formation World Tour
9                          The 1989 World Tour
10             The Mrs. Carter Show World Tour
11                     The Monster Ball Tour *
12                        Prismatic World Tour
13    Living Proof: The Farewell Tour ‡[21][a]
14                            Confessions Tour
15                   The Truth About Love Tour
16                          Born This Way Ball
17                            Rebel Heart Tour
18                             Adele Live 2016
19                                The Red Tour
Name: tour_title, dtype: object

0                  The Eras Tour 
1          Renaissance World Tour
2                         Sticky 
3     Beautiful Trauma World Tour
4         Reputation Stadium Tour
5                   The MDNA Tour
6       Taking Chances World Tour
7                Summer Carnival 
8        The Formation World Tour
9             The 1989 World Tour
10                        The Mrs
11         The Monster Ball Tour 
12           Prismatic World Tour
13                   Living Proof
14               Confessions Tour
15      The Truth About Love Tour
16             Born This Way Ball
17               Rebel Heart Tour
18                Adele Live 2016
19                   The Red Tour
Name: tour_title, dtype: object

### Data Cleaning: Handling Missing Values in 'peak'

There are 9 rows with missing data in the 'peak' column. Since I cannot confirm whether these albums actually failed to reach an peak or if the information is simply unavailable due to a lack of historical records, I have decided to leave them as 'N/A' to avoid introducing any inaccurate bias into the dataset.

In [50]:
tour_df['peak'].isna().sum()

np.int64(11)

### Data Cleaning: Handling Missing Values in 'all_time_peak'

There are 14 rows with missing data in the 'all_time_peak' column. Since I cannot confirm whether these albums actually failed to reach an all-time peak or if the information is simply unavailable due to a lack of historical records, I have decided to leave them as 'N/A' to avoid introducing any inaccurate bias into the dataset.

In [51]:
tour_df['all_time_peak'].isna().sum()

np.int64(14)